# 실습 7: 학습용과 시험용으로 나누기
- 상황: 아직 검사하지 않은 흐름의 결과를 맞혀보려 한다
- 목표: 답을 아는 기록과 모르는 척할 기록을 나눈다

## Step 0. 정제본 불러오기

In [1]:
import pandas as pd

df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

print(df.shape)
print(df["result"].value_counts())

(1567, 51)
result
양품    1463
불량     104
Name: count, dtype: int64


---

## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 모델을 만들 때 쓰는 말

| 말 | 뜻 |
|---|---|
| 지도학습 | 답이 붙어 있는 기록으로 규칙을 찾게 하는 방식. 우리 데이터의 검사 결과가 그 답이다 |
| 분류 | 둘 중 어느 쪽인지 맞히는 문제. 양품이냐 불량이냐 |
| 학습용 | 답을 보여주고 규칙을 찾게 할 몫 |
| 시험용 | 답을 숨겨두고 실력을 재는 데 쓸 몫 |
| 과적합 | 학습용을 통째로 외워버려 처음 보는 기록은 못 맞히는 상태 |
| 일반화 | 그 반대. 처음 보는 기록에도 통하는 상태. 우리가 원하는 것 |
| 클래스 불균형 | 한쪽이 드문 상태. 여기서는 불량이 약 6.6%뿐이다 |
| 층화추출 | 나눌 때 드문 쪽 비율을 양쪽에 똑같이 맞춰주는 방식 |

---

## Step 2. 남은 빈칸 채우기

In [2]:
sensor_cols = df.columns.drop("result")

print("[채우기 전]")
na_before = df[sensor_cols].isna()
n_sensor_cols = len(sensor_cols)
cols_with_na = na_before.sum()
n_cols_with_na = int((cols_with_na > 0).sum())
total_cells = int(na_before.size)
n_na_cells = int(na_before.sum().sum())
pct_na = n_na_cells / total_cells * 100

print(f"센서 열: {n_sensor_cols} 개")
print(f"빈칸이 있는 열: {n_cols_with_na} 개")
print(f"빈칸 총 개수: {n_na_cells} / 전체 칸: {total_cells}")
print(f"전체 칸 중 비율: {pct_na:.3f} %")
print()
print("빈칸이 많은 열 5개:")
print(cols_with_na[cols_with_na > 0].sort_values(ascending=False).head(5))

df[sensor_cols] = df[sensor_cols].fillna(df[sensor_cols].median())

print()
print("[채운 뒤]")
n_sensor_na_after = int(df[sensor_cols].isna().sum().sum())
n_total_na_after = int(df.isna().sum().sum())
print(f"센서 열 빈칸: {n_sensor_na_after} 개")
print(f"표 전체 빈칸: {n_total_na_after} 개")
print(f"result 열 그대로: {df['result'].value_counts().to_dict()}")
print(f"행/열: {df.shape}")

[채우기 전]
센서 열: 50 개
빈칸이 있는 열: 49 개
빈칸 총 개수: 1559 / 전체 칸: 78350
전체 칸 중 비율: 1.990 %

빈칸이 많은 열 5개:
sensor_520    715
sensor_570    273
sensor_558    260
sensor_091     51
sensor_077     24
dtype: int64

[채운 뒤]
센서 열 빈칸: 0 개
표 전체 빈칸: 0 개
result 열 그대로: {'양품': 1463, '불량': 104}
행/열: (1567, 51)


---

## Step 3. 정답표를 숫자로 바꾸기

In [3]:
# result 열은 "양품"·"불량"이라는 글자다. 모델은 글자로 학습하지 못한다.
# == 로 비교하면 참/거짓이 되고, astype(int)가 참을 1, 거짓을 0으로 바꾼다
df["불량여부"] = (df["result"] == "불량").astype(int)

# 1의 개수가 앞에서 본 불량 건수와 같아야 한다
print(df["불량여부"].value_counts())

불량여부
0    1463
1     104
Name: count, dtype: int64


---

## Step 4. 입력과 정답으로 가르기

In [4]:
# 입력 - 센서 열만. result 와 불량여부는 절대 들어가면 안 된다
센서열 = [c for c in df.columns if c.startswith("sensor_")]
X = df[센서열]

# 정답 - 맞혀야 할 것
y = df["불량여부"]

print("입력:", X.shape)
print("정답:", y.shape)

입력: (1567, 50)
정답: (1567,)


---

## Step 5. 학습용과 시험용으로 나누기

In [5]:
# train_test_split - 표를 학습용과 시험용 두 몫으로 갈라준다
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 시험용으로 떼어둘 비율 (20%)
    random_state=42,      # 무작위로 섞되, 다시 실행해도 같게 나오도록 고정
    stratify=y            # 불량 비율을 양쪽에 똑같이 맞춰서 나눈다 (층화추출)
)

print("학습용:", X_train.shape)
print("시험용:", X_test.shape)

학습용: (1253, 50)
시험용: (314, 50)


---

### 문법 노트 - 나누기

| 쓴 것 | 하는 일 | 왜 여기 쓰나 |
|---|---|---|
| train_test_split(X, y) | 입력과 정답을 같은 기준으로 두 몫씩 갈라준다 | 답을 숨겨둔 몫이 있어야 실력을 잰다 |
| test_size=0.2 | 시험용으로 뗄 비율 | 20%면 300건 정도 남는다 |
| random_state=42 | 섞는 방식을 고정 | 안 넣으면 돌릴 때마다 결과가 달라져 비교가 안 된다 |
| stratify=y | 정답 비율을 양쪽에 맞춰 나눈다 | 불량이 6.6%뿐이라 안 맞추면 한쪽에 몰린다 |

**돌려주는 것이 네 덩어리인 순서에 주의.**<br>
X_train, X_test, y_train, y_test 순서다. 입력 둘이 먼저, 정답 둘이 나중.<br>
순서를 바꿔 받으면 오류 없이 실행되면서 결과만 이상해진다.

# STEP6 : 제대로 나뉘었는지 확인하기

In [6]:
overall_rate = df["불량여부"].mean() * 100

rows = {
    "X_train": (X_train.shape[0], int(y_train.sum())),
    "X_test": (X_test.shape[0], int(y_test.sum())),
    "y_train": (len(y_train), int(y_train.sum())),
    "y_test": (len(y_test), int(y_test.sum())),
}

summary = pd.DataFrame(
    [
        {
            "이름": name,
            "행 수": n,
            "불량 건수": defect,
            "불량 비율(%)": round(defect / n * 100, 2),
            "원본 전체 불량 비율(%)": round(overall_rate, 2),
        }
        for name, (n, defect) in rows.items()
    ]
).set_index("이름")

summary

,행 수,불량 건수,불량 비율(%),원본 전체 불량 비율(%)
이름,,,,
X_train,1253,83,6.62,6.64
X_test,314,21,6.69,6.64
y_train,1253,83,6.62,6.64
y_test,314,21,6.69,6.64


[나눈 결과]<br>
학습용 : [1253]건 (불량 [83]건, [6.62]%)<br>
시험용 : [314]건 (불량 [21]건, [6.69]%)<br>
전체   : [1567]건 (불량 [104]건, [6.64]%)<br>
시험용 불량이 [21]건뿐이다.

## Step 7. 첫 예측 한 번 돌려보기

In [10]:
# LogisticRegression - 분류 문제에서 가장 기본이 되는 모델
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("예측 결과 개수:", len(y_pred))
print("불량이라고 예측한 건수:", int((y_pred == 1).sum()))
print("시험용 실제 불량 건수:", int(y_test.sum()))

예측 결과 개수: 314
불량이라고 예측한 건수: 0
시험용 실제 불량 건수: 21


c:\Users\subin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


---
## 직접 해보기 (도전) - 비율을 안 맞추고 나누면

- 상황: stratify 를 넣어야 한다고 배웠지만, 안 넣으면 실제로 얼마나 달라지는지는 모른다
- 할 일: stratify 없이 여러 번 나눠보고 시험용 불량 건수가 얼마나 흔들리는지 본다
- 결과물: 다섯 줄짜리 비교표 1개

In [9]:
# X_train, X_test, y_train, y_test 는 그대로 두고, 비교용 변수만 새로 만든다
rows = []
for rs in range(5):
    Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(
        X, y, test_size=0.2, random_state=rs, stratify=y
    )
    Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
        X, y, test_size=0.2, random_state=rs
    )
    rows.append({
        "random_state": rs,
        "stratify=y 불량 건수": int(ya_te.sum()),
        "stratify=y 불량 비율(%)": round(ya_te.mean() * 100, 2),
        "stratify 없음 불량 건수": int(yb_te.sum()),
        "stratify 없음 불량 비율(%)": round(yb_te.mean() * 100, 2),
    })

compare = pd.DataFrame(rows).set_index("random_state")
compare

,stratify=y 불량 건수,stratify=y 불량 비율(%),stratify 없음 불량 건수,stratify 없음 불량 비율(%)
random_state,,,,
0,21,6.69,13,4.14
1,21,6.69,20,6.37
2,21,6.69,20,6.37
3,21,6.69,19,6.05
4,21,6.69,26,8.28
